In [2]:
#!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.2 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
#Image Detection

import os
from pathlib import Path
from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator
import cv2
from google.colab.patches import cv2_imshow

# Load trained model
model_path = "/content/drive/MyDrive/Colab Notebooks/Projects/Deep Learning/DL_Capstone/EMV_Detection/YOLOv8_models/emv_detection_model.pt"
model = YOLO(model_path)

# Input and output folders
input_folder = "/content/drive/MyDrive/Colab Notebooks/Projects/Deep Learning/DL_Capstone/EMV_Detection/Demo/Image_input"
output_folder = "/content/drive/MyDrive/Colab Notebooks/Projects/Deep Learning/DL_Capstone/EMV_Detection/Demo/results"
os.makedirs(output_folder, exist_ok=True)

# Define custom colors
custom_colors = {
    0: (0, 0, 255),   # emergency = red
    1: (0, 255, 0)    # non_emergency = green
}

# Process all images in the folder
for filename in os.listdir(input_folder):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        image_path = os.path.join(input_folder, filename)

        # Run YOLO prediction
        results = model(image_path, conf=0.35, iou=0.45, verbose=False)

        # Draw results with custom colors
        for r in results:
            annotator = Annotator(r.orig_img.copy(), line_width=2, example=str(model.names))
            for box in r.boxes:
                cls = int(box.cls)
                label = model.names[cls]
                annotator.box_label(box.xyxy[0], label, color=custom_colors.get(cls, (255, 255, 255)))
            im = annotator.result()

            # Show one sample in Colab
            cv2_imshow(im)

            # Save result to output folder
            save_path = os.path.join(output_folder, filename)
            cv2.imwrite(save_path, im)

print(f"✅ All results saved in: {output_folder}")


Output hidden; open in https://colab.research.google.com to view.

In [28]:
from ultralytics import YOLO
import cv2

# Load your trained ambulance-only model
#model = YOLO("/content/runs/detect/train3/weights/best.pt")

# Input video path
video_path = "/content/drive/MyDrive/Colab Notebooks/Projects/Deep Learning/DL_Capstone/EMV_Detection/Demo/Demo5.mp4"
cap = cv2.VideoCapture(video_path)

# Output video writer
output_path = "/content/drive/MyDrive/Colab Notebooks/Projects/Deep Learning/DL_Capstone/EMV_Detection/Demo/Demo5_output.mp4"


fourcc = cv2.VideoWriter_fourcc(*"mp4v")
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

#Define custom colors (class_id: (B, G, R))
custom_colors = {
    0: (0, 0, 255),   # emergency = red
    1: (0, 255, 0)    # non_emergency = green
}

# Run inference frame by frame
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO prediction
    results = model(frame, conf=0.25, verbose=False)

    # Annotate frame with custom colors
    for r in results:
        annotator = Annotator(frame.copy(), line_width=2, example=str(model.names))
        for box in r.boxes:
            cls = int(box.cls)
            label = model.names[cls]
            annotator.box_label(box.xyxy[0], label, color=custom_colors.get(cls, (255, 255, 255)))
        annotated_frame = annotator.result()

    # Write frame to output video
    out.write(annotated_frame)

cap.release()
out.release()


print(f"✅ Inference complete! Saved at: {output_path}")

✅ Inference complete! Saved at: /content/drive/MyDrive/Colab Notebooks/Projects/Deep Learning/DL_Capstone/EMV_Detection/Demo/Demo5_output.mp4
